<a href="https://colab.research.google.com/github/sejuti009/Welcome-To-Data-Science-/blob/main/RecommendationEngineModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import libraries
import pandas as pd
import numpy as np

In [ ]:
# load data
rating_df = pd.read_csv('/content/ratings.csv')
movies_df = pd.read_csv('/content/movies.csv')

In [ ]:
# data processing
# extract year
movies_df['year'] = movies_df['title'].str.extract(r'\((\d{4})\)')

In [ ]:
# remove year from title
movies_df['title'] = movies_df['title'].str.replace(r'\(\d{4}\)','',regex=True).str.strip()

In [ ]:
# split genres
movies_df['genres'] = movies_df['genres'].str.split('|')

In [ ]:
# one-hot encoding using explode (FAST)
movies_exploded = movies_df.explode('genres')

In [ ]:
movies_onehot = pd.crosstab(movies_exploded['movieId'],movies_exploded['genres'])
# here crosstab function is used to create the contigency table

In [ ]:
# merge back
movies_copy = movies_df.merge(movies_onehot,on='movieId')

In [ ]:
# drop timestamp
rating_df.drop('timestamp',axis=1,inplace=True)

In [ ]:
# user input
user_input = [
    {'title': 'Scream', 'rating': 5},
    {'title': 'The Shining', 'rating': 5},
    {'title': 'Halloween', 'rating': 4.5},
    {'title': 'Nightmare on Elm Street', 'rating': 5},
    {'title': 'The Exorcist', 'rating': 4.5},
    {'title': 'Jumanji', 'rating': 2} # dislke non-horror
]

In [ ]:
movies_input = pd.DataFrame(user_input)

In [ ]:
# match title with dataset
input_movies = movies_df[movies_df['title'].isin(movies_input['title'])]

In [ ]:
# merge to get movieId
movies_input = pd.merge(input_movies,movies_input,on='title')

In [ ]:
# if no matches found
if movies_input.empty:
  raise ValueError('No matching movies found. Check titles!')

In [ ]:
# user profile
# get one-hot rows for input movies
user_movies = movies_copy[movies_copy['movieId'].isin(movies_input['movieId'])]

In [ ]:
# reset indew
user_movies = user_movies.reset_index(drop=True)
movies_input = movies_input.reset_index(drop=True)

In [ ]:
# create genre matrix
user_genres = user_movies.drop(columns=['movieId','title','genres','year'])

In [ ]:
# compute user profile
user_profile = user_genres.T.dot(movies_input['rating'])

In [ ]:
# recommendation
# full gerne table
genre_table = movies_copy.set_index('movieId')
genre_table = genre_table.drop(columns=['title','genres','year'])

In [ ]:
# compute scores
recommendation_scores = genre_table.dot(user_profile)

In [ ]:
# normalize (avoid divide by zeros)
# this works if user gives all the ratings of the movies as 0
if user_profile.sum() != 0:
  recommendation_scores = recommendation_scores / user_profile.sum()

In [ ]:
# sort recommendation
recommendation_scores = recommendation_scores.sort_values(ascending=False)

In [ ]:
# top 20 movies IDs
top_movies = recommendation_scores.head(20).index

In [ ]:
# final recommendation table
recommendation_table = movies_df[movies_df.isin(top_movies)]

In [ ]:
print(recommendation_table[['title','year']])

     title year
0      NaN  NaN
1      NaN  NaN
2      NaN  NaN
3      NaN  NaN
4      NaN  NaN
...    ...  ...
9737   NaN  NaN
9738   NaN  NaN
9739   NaN  NaN
9740   NaN  NaN
9741   NaN  NaN

[9742 rows x 2 columns]
